# Speech-Based Cognitive Screening — Notebook 01
## Experiment E1 · Transcript markers + speaker-independent baseline

**Team:** A. R. Nitesh Kumar (23BAI0108) · Harshini J. S. (23BAI0166) · Hari Prasath U. (23BAI0038)  


**What this does.** Parses the DementiaBank **Pitt Cookie-Theft** transcripts, extracts clinical
speech–language markers, checks they separate the groups, and trains a baseline classifier that is
evaluated the **correct** way — no participant ever appears in both train and test.

**Why it matters.** This is the *floor*. Every later model (concept bottleneck, wav2vec2, fusion)
must beat the AUC this notebook reports. **No audio is used here.**

▶ **Just press Runtime → Run all, and upload `Pitt.zip` when the button appears in Step 1.**


## 0 · Install dependencies

In [ ]:
!pip install pylangacq scikit-learn pandas pyarrow --quiet
import re, glob, os, numpy as np, pandas as pd
import pylangacq
print("pylangacq", pylangacq.__version__, "— setup OK")

## 1 · Upload `Pitt.zip`  ⬅ the only manual step

Run this cell. A **"Choose Files"** button appears — pick the `Pitt.zip` you downloaded from
TalkBank. It uploads, unzips, and locates the transcript folders automatically, however the zip is
nested. (If you have already uploaded it via the left Files panel, the cell will find it and skip
the button.)


In [ ]:
# --- get Pitt.zip into the session (button if not already present) ---
zip_candidates = [f for f in glob.glob("**/*.zip", recursive=True)
                  if "pitt" in os.path.basename(f).lower()]

if not zip_candidates:
    try:
        from google.colab import files
        print("Select your Pitt.zip …")
        up = files.upload()                       # <-- Choose Files button
        zip_candidates = [n for n in up.keys() if n.lower().endswith(".zip")]
    except Exception as e:
        print("Not on Colab or upload skipped:", e)

assert zip_candidates, "No Pitt.zip found. Re-run this cell and choose the file."
ZIP = zip_candidates[0]
print("Using zip:", ZIP)

# --- unzip ---
os.system(f'unzip -o "{ZIP}" -d pitt_data > /dev/null 2>&1')

# --- locate the folder that directly contains Control/ and Dementia/ (robust to nesting) ---
cha_files = [f for f in glob.glob("**/cookie/*.cha", recursive=True)
             if os.sep+"Control"+os.sep in f or os.sep+"Dementia"+os.sep in f]
print("Found", len(cha_files), "cookie transcripts")   # expect 552

def find_root(files):
    for f in files:
        p = f
        for _ in range(6):
            p = os.path.dirname(p)
            if os.path.isdir(os.path.join(p, "Control")) and os.path.isdir(os.path.join(p, "Dementia")):
                return p
    return None

assert cha_files, "Unzip produced no .cha files — check the file you uploaded is the Pitt transcript zip."
ROOT = find_root(cha_files)
assert ROOT, "Could not find a folder containing both Control/ and Dementia/."
print("ROOT =", ROOT)
print("  Control cookie files :", len(glob.glob(f"{ROOT}/Control/cookie/*.cha")))
print("  Dementia cookie files:", len(glob.glob(f"{ROOT}/Dementia/cookie/*.cha")))

## 2 · Parse the `@ID` header (labels)

Each Pitt file has a header like  
`@ID: eng|Pitt|PAR|57;|male|ProbableAD||Participant|18||`  
giving **age, sex, diagnosis, MMSE**. We read it with a regex.


In [ ]:
ID_RE = re.compile(r'@ID:\s*[^|]*\|[^|]*\|PAR\|([^|]*)\|([^|]*)\|([^|]*)\|[^|]*\|[^|]*\|([^|]*)\|')

def parse_header(path):
    txt = open(path, encoding="utf-8").read()
    m = ID_RE.search(txt)
    age = sex = dx = mmse = None
    if m:
        a, s, d, mm = m.groups()
        a = a.replace(";", "").strip(); age = int(a) if a.isdigit() else np.nan
        sex = s.strip() or None
        dx  = d.strip() or None
        mm  = mm.strip(); mmse = int(mm) if mm.isdigit() else np.nan
    return age, sex, dx, mmse

print("header parser ready →", parse_header(cha_files[0]))

## 3 · Marker extractor

Computes transcript-only markers from the **participant (PAR)** tier: lexical diversity
(TTR, MATTR), pronoun/noun/verb ratios, utterance length, pause statistics (from the millisecond
timestamps in the file), and speech rate. POS tags and timestamps come straight from the CHAT
file, so no external tagger is needed.

> ✅ **Filled-pause fix applied.** CHAT writes fillers as separate `&-um` / `&-uh` tokens *and*
> attaches them to words as `&=...`. Earlier this read 0 for every file. Here we count fillers from
> the raw utterance text with a regex, which recovers the real rate. Spot-check `001-0.cha` (it
> clearly contains `&-um`) to confirm a non-zero value.


In [ ]:
PUNCT = {".", "?", "!", ",", ";", ":"}
FILLER_RE = re.compile(r'&[-=]?(um|uh|er|ah|mm|hmm|eh)\b', re.IGNORECASE)

def real_word(tok):
    w = (tok.word or "").strip()
    if not w or w in PUNCT: return None
    if w.startswith("&"): return None            # filler / event codes are not lexical words
    if w.startswith("xxx") or w == "www": return None
    return w.lower()

def extract_one(path):
    chat = pylangacq.read_chat(path)
    raw_text = open(path, encoding="utf-8").read()
    filled = len(FILLER_RE.findall(raw_text))    # <-- robust filled-pause count

    words, pos_tags, utts = [], [], []
    total_speech_ms, n_utt, prev_end = 0, 0, None
    pauses = []

    for u in chat.utterances():
        if u.participant != "PAR":
            continue
        n_utt += 1
        if u.time_marks:
            s, e = u.time_marks
            total_speech_ms += max(0, e - s)
            if prev_end is not None and s > prev_end:
                pauses.append(s - prev_end)
            prev_end = e
        utt_words = []
        for t in u.tokens:
            w = real_word(t)
            if w:
                words.append(w); utt_words.append(w)
                pos_tags.append((t.pos or "").lower())
        if utt_words:
            utts.append(utt_words)

    n = len(words)
    if n == 0:
        return None
    pos = pd.Series(pos_tags)
    nouns   = int((pos == "noun").sum())
    verbs   = int((pos == "verb").sum())
    prons   = int((pos == "pron").sum())
    content = int(pos.isin(["noun", "verb", "adj", "adv"]).sum())
    dur_min = total_speech_ms / 60000.0 if total_speech_ms > 0 else np.nan
    pauses  = np.array(pauses, float)

    W = 25   # MATTR window
    if n >= W:
        mattr = float(np.mean([len(set(words[i:i+W]))/W for i in range(0, n-W+1)]))
    else:
        mattr = len(set(words)) / n

    return {
        "n_tokens": n, "n_utterances": n_utt,
        "ttr": len(set(words))/n, "mattr": mattr,
        "mlu_words": n/max(1, len(utts)),
        "pronoun_ratio": prons/n, "noun_ratio": nouns/n, "verb_ratio": verbs/n,
        "pronoun_noun_ratio": prons/max(1, nouns),
        "noun_verb_ratio": nouns/max(1, verbs),
        "lexical_density": content/n,
        "filled_pause_rate": filled/n,
        "speech_rate_wpm": (n/dur_min) if dur_min and dur_min > 0 else np.nan,
        "n_pauses": len(pauses),
        "mean_pause_ms": float(np.mean(pauses)) if len(pauses) else np.nan,
        "long_pause_ratio": float((pauses > 2000).mean()) if len(pauses) else np.nan,
        "total_speech_s": total_speech_ms/1000.0,
    }

print("extractor ready — filled-pause example:",
      round(extract_one(cha_files[0])["filled_pause_rate"], 4))

## 4 · Build the feature table

**The single most important line here is the participant ID.** Files like `001-0`, `001-1` are the
*same person* at different visits, so they must never be split across train and test. We also make
the ID **group-aware** (`Control-172` ≠ `Dementia-172`), because the prefix `172` occurs in both
groups and is not the same individual.


In [ ]:
rows = []
for grp in ["Control", "Dementia"]:
    for path in sorted(glob.glob(f"{ROOT}/{grp}/cookie/*.cha")):
        base = os.path.basename(path)[:-4]          # e.g. 001-0
        pid  = f"{grp}-{base.split('-')[0]}"        # group-aware participant id
        age, sex, dx, mmse = parse_header(path)
        feats = extract_one(path)
        if feats is None:
            continue
        rec = {"file": base, "participant_id": pid, "group": grp,
               "dx": dx, "age": age, "sex": sex, "mmse": mmse}
        rec.update(feats)
        rows.append(rec)

assert len(rows) > 0, f"No rows extracted — check ROOT ({ROOT!r}) and the counts printed in Step 1."

df = pd.DataFrame(rows)
df.to_parquet("pitt_cookie_features.parquet", index=False)
df.to_csv("pitt_cookie_features.csv", index=False)
print(f"{len(df)} recordings · {df.participant_id.nunique()} participants")
print("\nDiagnosis distribution:")
print(df.dx.value_counts())
df.head()

## 5 · Do the markers separate the groups?

Sanity-check the biology before any model. Expected from the clinical literature: AD shows
**higher** pronoun-to-noun ratio and **longer** pauses, and **lower** MLU. If these don't hold,
extraction is wrong.


In [ ]:
d = df[df.dx.isin(["Control", "ProbableAD"])].copy()
d["label"] = (d.dx == "ProbableAD").astype(int)

markers = ["mattr","pronoun_noun_ratio","mlu_words","lexical_density",
           "filled_pause_rate","speech_rate_wpm","long_pause_ratio","mean_pause_ms","ttr"]

summary = d.groupby("dx")[markers].mean().T
summary["direction"] = np.where(summary["ProbableAD"] > summary["Control"], "AD higher", "AD lower")
print(summary.round(3))

## 6 · Baseline classifier — done correctly

`StratifiedGroupKFold` keeps each participant entirely inside one fold; the `assert` proves no
leakage. We report **mean ± std** across 5 folds — never a single split. **This AUC is the number
every future model must beat.**


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

feat_cols = ["mattr","pronoun_noun_ratio","mlu_words","lexical_density",
             "filled_pause_rate","speech_rate_wpm","long_pause_ratio",
             "mean_pause_ms","ttr","noun_verb_ratio"]

X = d[feat_cols].values
y = d.label.values
groups = d.participant_id.values

models = {
    "LogReg":       make_pipeline(SimpleImputer(), StandardScaler(), LogisticRegression(max_iter=1000)),
    "RandomForest": make_pipeline(SimpleImputer(), RandomForestClassifier(n_estimators=300, random_state=0)),
}

print(f"Speaker-independent 5-fold CV · {len(d)} recordings · {d.participant_id.nunique()} participants\n")
for name, clf in models.items():
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    aucs, f1s, accs = [], [], []
    for tr, te in sgkf.split(X, y, groups):
        assert not (set(groups[tr]) & set(groups[te])), "LEAKAGE!"
        clf.fit(X[tr], y[tr])
        p = clf.predict_proba(X[te])[:, 1]
        pred = (p >= 0.5).astype(int)
        aucs.append(roc_auc_score(y[te], p))
        f1s.append(f1_score(y[te], pred))
        accs.append(accuracy_score(y[te], pred))
    print(f"{name:14s} AUC {np.mean(aucs):.3f}±{np.std(aucs):.3f}   "
          f"F1 {np.mean(f1s):.3f}±{np.std(f1s):.3f}   "
          f"Acc {np.mean(accs):.3f}±{np.std(accs):.3f}")

## 7 · (Optional) The harder task — MCI vs Control

AD vs Control is the easy problem. The clinically valuable one is catching **MCI**, the earliest
stage. Same code, different label filter. Expect lower numbers — that is the point, and it is where
a novel contribution lives.


In [ ]:
m = df[df.dx.isin(["Control", "MCI"])].copy()
m["label"] = (m.dx == "MCI").astype(int)
print("MCI task:", (m.label==1).sum(), "MCI vs", (m.label==0).sum(), "Control",
      "·", m.participant_id.nunique(), "participants")

Xm, ym, gm = m[feat_cols].values, m.label.values, m.participant_id.values
clf = make_pipeline(SimpleImputer(), StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
aucs = []
for tr, te in sgkf.split(Xm, ym, gm):
    assert not (set(gm[tr]) & set(gm[te]))
    clf.fit(Xm[tr], ym[tr])
    aucs.append(roc_auc_score(ym[te], clf.predict_proba(Xm[te])[:,1]))
print(f"MCI vs Control — LogReg AUC {np.mean(aucs):.3f}±{np.std(aucs):.3f}")

## 8 · Download your outputs (optional)

Saves the feature table so later notebooks train off this small file instead of the raw data.


In [ ]:
try:
    from google.colab import files
    files.download("pitt_cookie_features.csv")
except Exception as e:
    print("Files saved in session:", os.path.abspath("pitt_cookie_features.parquet"))

## What you produced & what's next

- **`pitt_cookie_features.parquet` / `.csv`** — your reusable feature table (input to every later experiment).
- A **speaker-independent baseline** with honest mean ± std — the floor for the project.

### Next: Notebook 02 — the concept bottleneck (Deep Learning core)
Runs on the feature table you just made — no audio, no GPU. It routes these markers through a
transparent additive head and measures whether being interpretable costs accuracy: your headline
Deep Learning result.

**Audio (Experiment E2) comes later** — only when you add acoustic markers (pitch, jitter,
spectrogram CNN). It is not needed for the bottleneck.
